# DriveSaverBot — Google Colab

GitHub هو المصدر الأساسي، وColab هو بيئة التشغيل.
مجلد الحفظ الثابت: **Auto Downloads**.


In [ ]:
# 1) تجهيز البيئة
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install -U python-telegram-bot google-api-python-client google-auth yt-dlp requests Flask


In [ ]:
# 2) الإعداد الأولي — شغّل هذه الخلية أول مرة فقط
from google.colab import drive, auth
from getpass import getpass
from pathlib import Path
import json, os

DRIVE_ROOT = Path('/content/drive/MyDrive')
APP_DIR = DRIVE_ROOT / '.drive_saver_bot'
CONFIG_FILE = APP_DIR / 'config.json'
AUTH_MARKER = APP_DIR / 'google_auth_ready'

# ربط Drive. الموافقة تتم من Colab نفسه، وليس عبر مفاتيح Google API.
drive.mount('/content/drive', force_remount=False)
APP_DIR.mkdir(parents=True, exist_ok=True)

config = {}
if CONFIG_FILE.exists():
    try:
        config = json.loads(CONFIG_FILE.read_text(encoding='utf-8'))
    except Exception:
        config = {}

if not config.get('BOT_TOKEN'):
    config['BOT_TOKEN'] = getpass('🔐 أدخل توكن Telegram — سيُحفظ في إعداداتك ولن يُرفع إلى GitHub: ').strip()
    if not config['BOT_TOKEN']:
        raise RuntimeError('لم يتم إدخال توكن Telegram.')

if not config.get('BOT_PASSWORD'):
    config['BOT_PASSWORD'] = getpass('🔑 أدخل كلمة مرور البوت الحالية: ').strip()
    if not config['BOT_PASSWORD']:
        raise RuntimeError('كلمة مرور البوت مطلوبة للحفاظ على حماية البوت.')

# مصادقة Google API تُجرى في الإعداد الأولي.
if not AUTH_MARKER.exists():
    auth.authenticate_user()
    AUTH_MARKER.write_text('ready', encoding='utf-8')

config['DRIVE_FOLDER_ID'] = '1MG3ot_alJno3POUINMmPR5vvMLc7-3ZA'
CONFIG_FILE.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')

print('✅ الإعداد محفوظ.')
print('📁 مجلد الحفظ: Auto Downloads')
print('🔐 لن يُطلب التوكن مرة أخرى ما دام ملف الإعداد محفوظاً في Drive.')


In [ ]:
# 3) التشغيل الكامل — هذه هي الخلية التي تشغّل البوت
import os, json
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
APP_DIR = DRIVE_ROOT / '.drive_saver_bot'
CONFIG_FILE = APP_DIR / 'config.json'

config = json.loads(CONFIG_FILE.read_text(encoding='utf-8'))
os.environ['BOT_TOKEN'] = config['BOT_TOKEN']
os.environ['BOT_PASSWORD'] = config['BOT_PASSWORD']
os.environ['DRIVE_FOLDER_ID'] = config['DRIVE_FOLDER_ID']
os.environ['COLAB_AUTH'] = '1'

!rm -rf /content/drive-saver-bot
!git clone -q https://github.com/Hus-GPT/drive-saver-bot.git /content/drive-saver-bot
%cd /content/drive-saver-bot
!python -u bot.py
